In [23]:
import json
from datetime import datetime, timedelta
from pathlib import Path
import requests
import os

# File where review data is stored
data_file = Path("review_data.json")

# Spaced repetition intervals (in days) based on review count
REVIEW_INTERVALS = [1, 2, 4, 7, 15]  # up to 5 reviews

# Deepseek API configuration
DEEPSEEK_API_URL = os.getenv("DEEPSEEK_API_URL", "https://api.deepseek.com/chat/completions")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY","Your API key here")

class Question:
    """
    Represents a LeetCode question, its review history, answers, and suggestions.
    """
    def __init__(self, url: str, data: dict = None):
        self.url = url
        if data:
            self.review_dates = [datetime.fromisoformat(d) for d in data.get("review_dates", [])]
            self.answers = data.get("answers", [])
            self.suggestions = data.get("suggestions", [])
            self.next_review = datetime.fromisoformat(data.get("next_review")) if data.get("next_review") else None
        else:
            self.review_dates = []
            self.answers = []  # list of code strings
            self.suggestions = []  # list of suggestion texts
            self.next_review = None

    @property
    def review_count(self) -> int:
        return len(self.review_dates)

    def record_review(self, code_answer: str, date: datetime = None):
        """
        Record a review with the provided code answer, request suggestions, and schedule the next.
        """
        date = date or datetime.now()
        if self.review_count < 5:
            self.review_dates.append(date)
            self.answers.append(code_answer)
            suggestion = self.get_suggestion(code_answer)
            print(suggestion)
            self.suggestions.append(suggestion)
            self.schedule_next()
        else:
            print(f"Maximum reviews reached for {self.url}")

    def schedule_next(self):
        """
        Schedule next review based on the forgetting curve intervals.
        """
        count = self.review_count - 1  # index into REVIEW_INTERVALS
        if count < len(REVIEW_INTERVALS):
            interval_days = REVIEW_INTERVALS[count]
            self.next_review = self.review_dates[-1] + timedelta(days=interval_days)
        else:
            self.next_review = None

    def get_suggestion(self, new_code: str) -> str:
        """
        Compare new_code with previous submission using Deepseek and return improvement suggestion.
        """
        if not DEEPSEEK_API_KEY:
            return "API key not configured."
        payload = {
            "model": "deepseek-reasoner",
            "messages":[{
                "role": "user",
                "content": f"{new_code} Please review the code and give some quick suggestions"
        }],
            "question_url": self.url,
            "new_code": new_code,
            "previous_code": self.answers[-2] if self.review_count > 1 else ""
        }
        headers = {
            "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
            "Content-Type": "application/json"
        }
        try:
            resp = requests.post(DEEPSEEK_API_URL, headers=headers, json=payload)
            resp.raise_for_status()
            data = resp.json()
            # return data.get("suggestion", "No suggestion returned.")
            # print(data["choices"][0]["message"]["content"].strip())
            return data["choices"][0]["message"]["content"].strip()
        except Exception as e:
            return f"Error fetching suggestion: {e}"

    def to_dict(self) -> dict:
        return {
            "url": self.url,
            "review_dates": [d.isoformat() for d in self.review_dates],
            "answers": self.answers,
            "suggestions": self.suggestions,
            "next_review": self.next_review.isoformat() if self.next_review else None
        }

class ReviewManager:
    """
    Manages multiple questions and their schedules.
    """
    def __init__(self, data_path: Path = data_file):
        self.data_path = data_path
        self.questions = {}  # url -> Question
        self.load()

    def load(self):
        if self.data_path.exists():
            raw = json.loads(self.data_path.read_text())
            for url, info in raw.items():
                self.questions[url] = Question(url, info)

    def save(self):
        data = {url: q.to_dict() for url, q in self.questions.items()}
        self.data_path.write_text(json.dumps(data, indent=2))

    def add_question(self, url: str):
        if url not in self.questions:
            self.questions[url] = Question(url)

    def record_review(self, url: str, code_answer: str, date: datetime = None):
        if url not in self.questions:
            self.add_question(url)
        self.questions[url].record_review(code_answer, date)
        self.save()

    def due_today(self, max_count: int = 5) -> list:
        today = datetime.now().date()
        due = [q for q in self.questions.values() if q.next_review and q.next_review.date() <= today]
        due.sort(key=lambda q: q.next_review)
        return due[:max_count]

def codeExtract(pathToFile):
    with open(pathToFile, "r") as f:
        code_text = f.read()
    safe_code = json.dumps(pathToFile)
    lines = code_text.splitlines()
    chunk_size = 200  # lines per chunk
    chunks = ["\n".join(lines[i:i+chunk_size]) for i in range(0, len(lines), chunk_size)]
    return chunks

### Example usage:
this is to randomly select 5 question from your practiced list

In [ ]:
if __name__ == "__main__":
    manager = ReviewManager()
    due = manager.due_today()
    print("Questions due for review today:")
    for q in due:
        print(f"- {q.url} (reviewed {q.review_count} times, suggestion: {q.suggestions[-1]})")

### Example usage:
**url**: the url of solved question (right now one at a time)
**code**: the file where you wrote your most recent solution

In [ ]:
if __name__ == "__main__":
    manager = ReviewManager()
    # Example usage:
    url = "https://leetcode.com/problems/search-a-2d-matrix/"
    code = codeExtract('./repo/Search a 2D Matrix_ver1.py')
    manager.record_review(url, code)